# **Estudo - Data Science (Python)**

No seguinte problema, temos uma base de dados extensa com informações gerais adquiridas por um fornecedor
de crédito. O objetivo desse trabalho é fazer o tratamento e a análise desses dados com o intuito de obter
o padrão de características presentes nos clientes inadimplentes quanto a seus pagamentos de crédito bancário. Para isso, é necessário separar o problema em etapas e trata-las de forma organizada e precisa.

# **Primeira etapa: exploração de dados**

Inicialmente, é feita uma análise geral da estrutura desses dados, observando o formato deles, se há
lacunas vazias e etc.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/andre-marcos-perez/ebac-course-utils/develop/dataset/credito.csv', na_values='na')

In [ ]:
df.head(n=15)

df.shape # nos mostra a quantidade de linhas e colunas (linhas, colunas)

df[df['default'] == 0].shape[0] # quantidade de linhas relacionadas a clientes adimplentes (default = 0)

df[df['default'] == 1].shape[0] # quantidade de linhas relacionadas a clientes inadimplentes (default = 1)

qtd_total, _ = df.shape
qtd_adimplentes, _ = df[df['default'] == 0].shape
qtd_inadimplentes, _ = df[df['default'] == 1].shape

print(f"{round(100 * qtd_adimplentes / qtd_total, 2)}% dos clientes é adimplente")
print(f"{round(100 * qtd_inadimplentes / qtd_total, 2)}% dos clientes é inadimplente")

df.dtypes

df.select_dtypes('object').describe().transpose()

df.drop('id', axis=1).select_dtypes('number').describe().transpose()

df.isna().any()

def stats_dados_faltantes(df: pd.DataFrame) -> None:

  stats_dados_faltantes = []
  for col in df.columns:
    if df[col].isna().any():
      qtd, _ = df[df[col].isna()].shape
      total, _ = df.shape
      dict_dados_faltantes = {col: {'quantidade': qtd, "porcentagem": round(100 * qtd/total, 2)}}
      stats_dados_faltantes.append(dict_dados_faltantes)

  for stat in stats_dados_faltantes:
    print(stat)

stats_dados_faltantes(df=df)

stats_dados_faltantes(df=df[df['default'] == 0])

stats_dados_faltantes(df=df[df['default'] == 1])

# **Segunda etapa: tratamento dos dados**

Analizando as informações obtidas acima, é possível notar que a porcentagem dos dados faltantes de escolaridade, estado civil e salário anual é semelhante em adimplentes e inadimplentes. Portanto, deve ser seguro remover esses dados antes da análise. Outro ponto importante é que os valores das colunas 'limite_credito' e, 'valor_transacoes_12m' estão sendo tratados como 'objetos', enquanto deveriam ser tratados como 'floats'. Portanto, devemos corrigir esses erros antes de prosseguir com a análise.

In [ ]:
fn = lambda valor: float(valor.replace(".", "").replace(",", "."))

df['valor_transacoes_12m'] = df['valor_transacoes_12m'].apply(fn)
df['limite_credito'] = df['limite_credito'].apply(fn)

df.dtypes

df.select_dtypes('object').describe().transpose()

df.drop('id', axis=1).select_dtypes('number').describe().transpose()

df.dropna(inplace=True)

df.shape

df[df['default'] == 0].shape

df[df['default'] == 1].shape

qtd_total_novo, _ = df.shape
qtd_adimplentes_novo, _ = df[df['default'] == 0].shape
qtd_inadimplentes_novo, _ = df[df['default'] == 1].shape

print(f"A proporcão adimplentes ativos é de {round(100 * qtd_adimplentes / qtd_total, 2)}%")
print(f"A nova proporcão de clientes adimplentes é de {round(100 * qtd_adimplentes_novo / qtd_total_novo, 2)}%")
print("")
print(f"A proporcão clientes inadimplentes é de {round(100 * qtd_inadimplentes / qtd_total, 2)}%")
print(f"A nova proporcão de clientes inadimplentes é de {round(100 * qtd_inadimplentes_novo / qtd_total_novo, 2)}%")

Isso indica, portanto, que a proporção entre adimplentes e inadimplentes foi pouco alterada removendo
os clientes com dados faltantes.

# **Terceira etapa: visualização e análise dos dados**

Agora com os dados devidamente tratados, é possível plotar e analisar os gráficos relacionando as
variáveis independentes com a dependente.

In [ ]:
sns.set_style("whitegrid")

df_adimplente = df[df['default'] == 0]

df_inadimplente = df[df['default'] == 1]

df.select_dtypes('object').head(n=5)

In [ ]:
coluna = 'escolaridade'
titulos = ['Escolaridade dos Clientes', 'Escolaridade dos Clientes Adimplentes', 'Escolaridade dos Clientes Inadimplentes']

figura, eixos = plt.subplots(1, 3, figsize=(20, 5), sharex=True)
max_y = 0

for eixo, dataframe in enumerate([df, df_adimplente, df_inadimplente]):
    df_to_plot = dataframe[coluna].value_counts().reset_index()
    df_to_plot.columns = [coluna, 'frequencia_absoluta']
    df_to_plot.sort_values(by=[coluna], inplace=True)

    f = sns.barplot(data=df_to_plot, x=coluna, y='frequencia_absoluta', ax=eixos[eixo])
    f.set(title=titulos[eixo], xlabel=coluna.capitalize(), ylabel='Frequência Absoluta')
    
    f.set_xticklabels(labels=f.get_xticklabels(), rotation=90)

    _, max_y_f = f.get_ylim()
    max_y = max_y_f if max_y_f > max_y else max_y

for eixo in eixos:
    eixo.set(ylim=(0, max_y))

plt.show()

In [ ]:
coluna = 'salario_anual'
titulos = ['Escolaridade dos Clientes', 'Escolaridade dos Clientes Adimplentes', 'Escolaridade dos Clientes Inadimplentes']

figura, eixos = plt.subplots(1, 3, figsize=(20, 5), sharex=True)
max_y = 0

for eixo, dataframe in enumerate([df, df_adimplente, df_inadimplente]):
    df_to_plot = dataframe[coluna].value_counts().reset_index()
    df_to_plot.columns = [coluna, 'frequencia_absoluta']
    df_to_plot.sort_values(by=[coluna], inplace=True)

    f = sns.barplot(data=df_to_plot, x=coluna, y='frequencia_absoluta', ax=eixos[eixo])
    f.set(title=titulos[eixo], xlabel=coluna.capitalize(), ylabel='Frequência Absoluta')
    
    f.set_xticklabels(labels=f.get_xticklabels(), rotation=90)

    _, max_y_f = f.get_ylim()
    max_y = max_y_f if max_y_f > max_y else max_y

for eixo in eixos:
    eixo.set(ylim=(0, max_y))

plt.show()

In [ ]:
df.drop(['id', 'default'], axis=1).select_dtypes('number').head(n=5)

coluna = 'qtd_transacoes_12m'
titulos = ['Qtd. de Transações no Último Ano', 'Qtd. de Transações no Último Ano de Adimplentes', 'Qtd. de Transações no Último Ano de Inadimplentes']

eixo = 0
max_y = 0
figura, eixos = plt.subplots(1,3, figsize=(20, 5), sharex=True)

for dataframe in [df, df_adimplente, df_inadimplente]:

  f = sns.histplot(x=coluna, data=dataframe, stat='count', ax=eixos[eixo])
  f.set(title=titulos[eixo], xlabel=coluna.capitalize(), ylabel='Frequência Absoluta')

  _, max_y_f = f.get_ylim()
  max_y = max_y_f if max_y_f > max_y else max_y
  f.set(ylim=(0, max_y))

  eixo += 1

figura.show()

coluna = 'valor_transacoes_12m'
titulos = ['Valor das Transações no Último Ano', 'Valor das Transações no Último Ano de Adimplentes', 'Valor das Transações no Último Ano de Inadimplentes']

eixo = 0
max_y = 0
figura, eixos = plt.subplots(1,3, figsize=(20, 5), sharex=True)

for dataframe in [df, df_adimplente, df_inadimplente]:

  f = sns.histplot(x=coluna, data=dataframe, stat='count', ax=eixos[eixo])
  f.set(title=titulos[eixo], xlabel=coluna.capitalize(), ylabel='Frequência Absoluta')

  _, max_y_f = f.get_ylim()
  max_y = max_y_f if max_y_f > max_y else max_y
  f.set(ylim=(0, max_y))

  eixo += 1

figura.show()

f = sns.relplot(x='valor_transacoes_12m', y='qtd_transacoes_12m', data=df, hue='default')
_ = f.set(
    title='Relação entre Valor e Quantidade de Transações no Último Ano',
    xlabel='Valor das Transações no Último Ano',
    ylabel='Quantidade das Transações no Último Ano'
  )

# Resumo de insights gerados pela análise: #

Observando os dados e as informações estatísticas deles, é possível relacionar diretamente a
adimplência/inadimplência dos clientes com a quantidade de transações e o valor de transações do último ano.

1.   De acordo com a análise, clientes que fazem menos de cerca de 80 transações por ano tendem a uma maior inadimplência.
2.   Da mesma forma, clientes que tiveram um valor total de transações inferior a 10 mil reais tendem a serem também inadimplentes.
3.   Em geral, a inadimplência se torna mais provável quando os dois casos acima ocorrem para um mesmo cliente. O caso em que é mais comum que um cliente seja inadimplente é aquele em que o cliente transacionou menos de 80 vezes no último ano, e as transações totais somam menos que 10 mil reais.